# Data Exploration and Augmentation Pipeline — Complete Guide

This notebook walks through the full augmentation pipeline from first principles.
Each section introduces one new concept, building on the previous, from the simplest
(what does a raw card look like?) to the most abstract (what does normalization do
to pixel values and why does it break visualization?).

**Sections:**
1. Load a raw card image and understand its labels
2. Resize: standardizing input size for ResNet18
3. Each transform in isolation — see what each step actually does
4. The full pipeline: all transforms together
5. Why order matters: `AddRandomBackground` must follow `ShiftScaleRotate`
6. Randomness in action: the `p=` parameter across many runs
7. Training vs. validation pipeline: randomness vs. reproducibility
8. Normalization: why you cannot display raw model-ready tensors directly

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
import sys

# Add project root to sys.path so we can import from src/
sys.path.append(os.path.abspath('..'))

from src.utils.augmentations import (
    get_spatial_color_transforms,
    get_train_transforms,
    get_val_transforms,
    AddRandomBackground,
)

# ImageNet normalization constants — defined once and reused throughout
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

%matplotlib inline

---
## 1. Load a Raw Card Image and Understand its Labels

Before touching any augmentation code, let's understand the raw data.
Every image in `data/raw/` is named `{color}_{shape}_{number}_{shading}.jpg`.
The label for each of the four features is encoded directly in the filename — there is no separate CSV or annotation file.

The code below mirrors exactly what `SetCardDataset.__getitem__` does on every sample load:
it reads the image, converts BGR to RGB, then parses the filename to extract four integer class labels.

In [ ]:
RAW_DIR     = '../data/raw'
SAMPLE_IMAGE = 'red_diamond_1_solid.jpg'
img_path     = os.path.join(RAW_DIR, SAMPLE_IMAGE)

# OpenCV loads images in BGR order — always convert to RGB immediately
image_bgr = cv2.imread(img_path)
if image_bgr is None:
    raise FileNotFoundError(f'Image not found: {img_path}. Ensure data/raw/ is populated.')
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# These mappings live in set_card_data_pipeline.py — duplicated here for reference
LABEL_MAPS = {
    'color':   {'red': 0, 'green': 1, 'purple': 2},
    'shape':   {'diamond': 0, 'squiggle': 1, 'oval': 2},
    'number':  {'1': 0, '2': 1, '3': 2},
    'shading': {'solid': 0, 'striped': 1, 'open': 2},
}

# Parse the filename exactly as SetCardDataset does
parts = os.path.splitext(SAMPLE_IMAGE)[0].split('_')
color_str, shape_str, number_str, shading_str = parts[0], parts[1], parts[2], parts[3]
parsed = {
    'color':   (color_str,   LABEL_MAPS['color'][color_str]),
    'shape':   (shape_str,   LABEL_MAPS['shape'][shape_str]),
    'number':  (number_str,  LABEL_MAPS['number'][number_str]),
    'shading': (shading_str, LABEL_MAPS['shading'][shading_str]),
}

plt.figure(figsize=(4, 4))
plt.imshow(image_rgb)
plt.title(f'File: {SAMPLE_IMAGE}')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Image shape: {image_rgb.shape}  (Height x Width x Channels)')
print(f'Pixel dtype: {image_rgb.dtype}  (uint8 — values 0 to 255)')
print()
print('Labels parsed from filename:')
for feature, (string_val, class_idx) in parsed.items():
    print(f'  {feature:8s}: {string_val:10s} -> class index {class_idx}')
print()
print('These integer class indices are what CrossEntropyLoss and F1Score receive.')
print('The string values ("red", "diamond", etc.) never reach the model.')

---
## 2. Resize: Standardizing Input Size for ResNet18

The very first step in both the training and validation pipelines is `A.Resize(224, 224)`.
ResNet18 (our backbone) was designed for 224×224 input. Without this step, different
raw images with different dimensions would produce tensors of different shapes, which cannot
be stacked into a batch.

This resize was missing from the original code and is tracked as **Issue #1** (now fixed).
This cell demonstrates the effect directly.

In [ ]:
resize_transform = A.Resize(224, 224)
image_resized    = resize_transform(image=image_rgb)['image']

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_rgb)
axes[0].set_title(f'Original: {image_rgb.shape[1]}x{image_rgb.shape[0]} px')
axes[0].axis('off')
axes[1].imshow(image_resized)
axes[1].set_title('After A.Resize(224, 224): 224x224 px')
axes[1].axis('off')
fig.suptitle('Step 1 of Every Pipeline: Resize to ResNet18 Input Size', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Original shape: {image_rgb.shape}')
print(f'Resized shape:  {image_resized.shape}')
print()
print('All subsequent transforms and visualizations in this notebook')
print('use image_resized (224x224) as their starting point.')

---
## 3. Each Transform in Isolation

The full pipeline applies five transforms, but it is hard to tell which one caused which change
when they all run together. This section applies each transform **one at a time** with `p=1.0`
(forced always-on) so you can see exactly what each step contributes.

In production the probabilities are lower (0.7 or 0.3) — this ensures each augmented image
is a random combination of some, but not necessarily all, of these effects.

> Note: `AddRandomBackground` runs at `p=1.0` in production too — it is always needed
> to clean up the dark padding that `ShiftScaleRotate` introduces.

In [ ]:
# Each entry: (display name, transform forced to p=1.0)
isolated_transforms = [
    (
        '1. ShiftScaleRotate\n   shift<=5%, scale<=10%, rotate<=45 deg\n   Simulates misalignment / zoom / tilt',
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=45,
                           border_mode=cv2.BORDER_CONSTANT, value=0, p=1.0)
    ),
    (
        '2. AddRandomBackground\n   Replaces dark/black pixels with\n   solid colors or random noise',
        AddRandomBackground(p=1.0)
    ),
    (
        '3. RandomBrightnessContrast\n   brightness/contrast +-20%\n   Simulates different lighting',
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0)
    ),
    (
        '4. HueSaturationValue\n   hue STRICTLY +-15 deg\n   Prevents color labels from corrupting',
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=30, p=1.0)
    ),
    (
        '5. GaussianBlur\n   kernel up to 3x3\n   Simulates camera blur / motion',
        A.GaussianBlur(blur_limit=3, p=1.0)
    ),
]

NUM_VARIANTS = 4
fig, axes = plt.subplots(len(isolated_transforms), NUM_VARIANTS + 1, figsize=(20, 24))

for row, (name, transform) in enumerate(isolated_transforms):
    axes[row, 0].imshow(image_resized)
    if row == 0:
        axes[row, 0].set_title('Original', fontsize=11)
    axes[row, 0].set_ylabel(name, fontsize=8, rotation=0,
                             labelpad=190, va='center', ha='right')
    axes[row, 0].axis('off')
    for col in range(1, NUM_VARIANTS + 1):
        aug = transform(image=image_resized)['image']
        axes[row, col].imshow(aug)
        if row == 0:
            axes[row, col].set_title(f'Variant {col}', fontsize=11)
        axes[row, col].axis('off')

fig.suptitle('Each Transform in Isolation (p=1.0 forces always-on — production uses lower p values)',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. The Full Pipeline: All Transforms Together

Now that you have seen each transform individually, this is what the full
`get_spatial_color_transforms()` pipeline produces — all five transforms
running in sequence with their production `p=` values.

Each variant is unique because each transform independently rolls its dice.
This is what the bootstrap script (`bootstrap_dataset.py`) calls 300 times
per seed image to generate the training set.

In [ ]:
pipeline = get_spatial_color_transforms()

NUM_COLS = 4
NUM_ROWS = 2
fig, axes = plt.subplots(NUM_ROWS, NUM_COLS, figsize=(16, 8))
fig.suptitle('Full Pipeline: All 5 Transforms Running Together on ' + SAMPLE_IMAGE,
             fontsize=14)

for i, ax in enumerate(axes.flatten()):
    augmented = pipeline(image=image_resized)['image']
    ax.imshow(augmented)
    ax.set_title(f'Variant {i + 1}')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 5. Why Order Matters: `AddRandomBackground` Must Follow `ShiftScaleRotate`

The order of transforms in `A.Compose([...])` is not arbitrary.

`ShiftScaleRotate` with `border_mode=cv2.BORDER_CONSTANT, value=0` fills any area
outside the original image with **solid black (value 0)**. `AddRandomBackground` works
by detecting dark pixels (brightness < 30) and replacing them.

If the order is reversed:
- `AddRandomBackground` runs first — the original image may have some dark corners,
  but no black padding yet (rotation hasn't happened). It replaces those and finishes.
- `ShiftScaleRotate` runs second — introduces new black padding around the rotated card.
- Nobody is left to replace that new black padding. It stays black.

The grid below shows the same image through both orderings. Look at the card edges.

In [ ]:
shift_rotate = A.ShiftScaleRotate(
    shift_limit=0.05, scale_limit=0.1, rotate_limit=45,
    border_mode=cv2.BORDER_CONSTANT, value=0, p=1.0)
add_bg = AddRandomBackground(p=1.0)

correct_pipeline = A.Compose([shift_rotate, add_bg])  # ShiftRotate THEN AddBackground
wrong_pipeline   = A.Compose([add_bg, shift_rotate])  # AddBackground THEN ShiftRotate

NUM_VARIANTS = 4
fig, axes = plt.subplots(2, NUM_VARIANTS, figsize=(16, 8))

for i in range(NUM_VARIANTS):
    axes[0, i].imshow(correct_pipeline(image=image_resized)['image'])
    axes[1, i].imshow(wrong_pipeline(image=image_resized)['image'])
    axes[0, i].axis('off')
    axes[1, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Variant 1', fontsize=10)

axes[0, 0].set_ylabel('CORRECT\nShiftRotate -> AddBg', rotation=0,
                       labelpad=150, va='center', fontsize=10)
axes[1, 0].set_ylabel('WRONG\nAddBg -> ShiftRotate', rotation=0,
                       labelpad=150, va='center', fontsize=10)

fig.suptitle('Order Matters: AddRandomBackground MUST Run After ShiftScaleRotate', fontsize=13)
plt.tight_layout()
plt.show()

print('CORRECT: ShiftScaleRotate creates black padding -> AddRandomBackground replaces it.')
print('WRONG:   AddRandomBackground runs first (nothing to fix) -> ShiftScaleRotate creates')
print('         new black padding that stays black for the rest of the pipeline.')

---
## 6. Randomness in Action: The `p=` Parameter Across Many Runs

Every transform has a `p=` parameter. Each time the pipeline runs, each transform
rolls an independent random number. If the number is below `p`, the transform fires;
otherwise, the image passes through unchanged.

This means different runs produce images with very different amounts of change.
Some runs will have almost all transforms fire (heavily augmented); others may have
only one or two fire (barely changed from the original).

The grid below shows 50 runs on the same image. The bar chart and histogram below it
quantify the probabilities and the spread of how much each run changes the image.

In [ ]:
NUM_RUNS = 50
pipeline = get_spatial_color_transforms()

# Run the pipeline NUM_RUNS times and record how much each run changed the image
pixel_diffs = []
fig, axes = plt.subplots(5, 10, figsize=(20, 10))
for ax in axes.flatten():
    aug  = pipeline(image=image_resized)['image']
    diff = float(np.mean(np.abs(aug.astype(float) - image_resized.astype(float))))
    pixel_diffs.append(diff)
    ax.imshow(aug)
    ax.axis('off')

fig.suptitle(f'{NUM_RUNS} Augmented Versions of the Same Image — Notice the Variance',
             fontsize=13)
plt.tight_layout()
plt.show()

# Bar chart of p= values + histogram of observed pixel changes
transform_names = ['ShiftScaleRotate', 'AddRandomBackground',
                   'RandomBrightnessContrast', 'HueSaturationValue', 'GaussianBlur']
transform_probs = [0.7, 1.0, 0.7, 0.7, 0.3]
bar_colors = ['#4CAF50' if p == 1.0 else '#2196F3' if p >= 0.7 else '#FF9800'
              for p in transform_probs]

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 5))

bars = axes2[0].bar(transform_names, transform_probs, color=bar_colors)
axes2[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% line')
axes2[0].set_ylim(0, 1.15)
axes2[0].set_title('p= values: probability each transform fires per image', fontsize=11)
axes2[0].set_ylabel('Probability of firing')
axes2[0].tick_params(axis='x', rotation=30)
axes2[0].legend(fontsize=9)
for bar, p in zip(bars, transform_probs):
    axes2[0].text(bar.get_x() + bar.get_width() / 2, p + 0.02,
                  f'p={p}', ha='center', fontsize=9, fontweight='bold')

axes2[1].hist(pixel_diffs, bins=15, color='steelblue', edgecolor='black', alpha=0.8)
axes2[1].axvline(x=np.mean(pixel_diffs), color='red', linestyle='--',
                 label=f'Mean = {np.mean(pixel_diffs):.1f}')
axes2[1].set_title(f'Pixel change spread across {NUM_RUNS} pipeline runs', fontsize=11)
axes2[1].set_xlabel('Mean absolute pixel difference from original (0-255 scale)')
axes2[1].set_ylabel('Count')
axes2[1].legend()
plt.tight_layout()
plt.show()

print(f'Across {NUM_RUNS} runs of the same image:')
print(f'  Least changed: {min(pixel_diffs):.1f}  avg pixel delta (few transforms fired)')
print(f'  Most changed:  {max(pixel_diffs):.1f}  avg pixel delta (many transforms fired)')
print(f'  Mean change:   {np.mean(pixel_diffs):.1f}')

---
## 7. Training vs. Validation Pipeline: Randomness vs. Reproducibility

The training and validation pipelines are deliberately different:

- **Training** (`get_train_transforms`): Resize → all augmentations → Normalize → ToTensorV2  
  Every call produces a different image. The model sees a new random view of each card each epoch.

- **Validation** (`get_val_transforms`): Resize → Normalize → ToTensorV2  
  No augmentation. Every call on the same image produces **identical** output.
  This makes validation metrics comparable across epochs — the model always sees the same images.

Both pipelines produce normalized tensors, which cannot be displayed directly.
This cell applies a `denormalize()` step before showing the images.

In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for display. Input: torch.Tensor (C,H,W)."""
    arr = tensor.numpy().transpose(1, 2, 0)        # (C,H,W) -> (H,W,C)
    return (arr * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)

train_tf = get_train_transforms()
val_tf   = get_val_transforms()
NUM_COLS = 5

fig, axes = plt.subplots(2, NUM_COLS, figsize=(20, 8))

for i in range(NUM_COLS):
    train_img = denormalize(train_tf(image=image_resized)['image'])
    val_img   = denormalize(val_tf(image=image_resized)['image'])

    axes[0, i].imshow(train_img)
    axes[0, i].set_title(f'Run {i + 1}', fontsize=11)
    axes[0, i].axis('off')

    axes[1, i].imshow(val_img)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('TRAINING\n(random each run)', rotation=0,
                       labelpad=130, va='center', fontsize=10)
axes[1, 0].set_ylabel('VALIDATION\n(identical every run)', rotation=0,
                       labelpad=130, va='center', fontsize=10)

fig.suptitle('Training vs. Validation Pipeline — Both denormalized to [0, 1] for display',
             fontsize=13)
plt.tight_layout()
plt.show()

print('Training:   Resize -> augmentation -> Normalize -> ToTensorV2')
print('            Different result every call — model sees varied images each epoch.')
print()
print('Validation: Resize -> Normalize -> ToTensorV2')
print('            Same result every call — metrics are comparable across epochs.')

---
## 8. Normalization: Why You Cannot Display Raw Model-Ready Tensors

The last two steps of both pipelines are `A.Normalize` and `ToTensorV2`.
After these steps the image is no longer a viewable image — it is a model input.

**What `A.Normalize` does:**
```
normalized_pixel = (pixel / 255.0 - mean) / std
```
Using the ImageNet constants `mean=[0.485, 0.456, 0.406]` and `std=[0.229, 0.224, 0.225]`.
This shifts pixel values from the `[0, 255]` uint8 range to a `float32` range roughly `[-2.0, +2.5]`.

**Why these specific constants?** ResNet18 was pre-trained on ImageNet with these statistics.
Its early layer weights expect input in this scale. Feeding raw 0–255 values would put
activations on the wrong scale and degrade the pre-trained features.

**What `ToTensorV2` does:**
Reshapes from `(H, W, C)` (NumPy convention) to `(C, H, W)` (PyTorch convention) and
converts to `torch.Tensor` — without copying memory where possible.

**To display a normalized tensor**, reverse the operation:
```python
display = (tensor * std + mean).clip(0, 1)
```

In [ ]:
# Apply only val_transforms (no random augmentation) so the normalization effect is clear
val_tf = get_val_transforms()
tensor = val_tf(image=image_resized)['image']

numpy_chw = tensor.numpy()                         # torch.Tensor (C,H,W) -> numpy (C,H,W)
numpy_hwc = numpy_chw.transpose(1, 2, 0)           # (C,H,W) -> (H,W,C) for matplotlib
denormalized = (numpy_hwc * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)

print('--- Tensor properties after the full val pipeline ---')
print(f'Type:         {type(tensor).__name__}')
print(f'Shape:        {tensor.shape}  (C x H x W — channels-first, PyTorch convention)')
print(f'dtype:        {tensor.dtype}  (float32, not uint8)')
print(f'Value range:  [{tensor.min():.3f}, {tensor.max():.3f}]')
print(f'Original range was [0, 255]')
print()
print('Normalization per channel:  (pixel / 255 - mean) / std')
print(f'  mean = {IMAGENET_MEAN}  (R, G, B)')
print(f'  std  = {IMAGENET_STD}')
print()
print('To display:  (tensor_hwc * std + mean).clip(0, 1)')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image_resized)
axes[0].set_title('Original (resized)\ndtype: uint8  |  range: [0, 255]', fontsize=10)
axes[0].axis('off')

axes[1].imshow(numpy_hwc)   # matplotlib auto-clips negative values to 0 -> looks wrong
axes[1].set_title(f'Raw normalized tensor\ndtype: float32  |  range: [{tensor.min():.2f}, {tensor.max():.2f}]\n(matplotlib clips negatives -> distorted colors)',
                  fontsize=9)
axes[1].axis('off')

axes[2].imshow(denormalized)
axes[2].set_title('Denormalized for display\ndtype: float32  |  range: [0.0, 1.0]\n(apply this whenever displaying pipeline output)',
                  fontsize=9)
axes[2].axis('off')

fig.suptitle('Normalization: What Happens to Pixel Values Inside the Pipeline', fontsize=13)
plt.tight_layout()
plt.show()